In [ ]:
# STEP 0 — CONFIG
from pathlib import Path
import gc
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import fasttext
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Source CSVs stay on the Windows disk (same files the baseline used).
DATA_DIR = Path("/mnt/c/Users/bsarv/Fake Desktop/amz sentiment analysis/archive")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# fastText corpora + models live on the Linux disk.
# through /mnt/c is an order of magnitude slower.
FT_DIR = Path.home() / "ft"
FT_DIR.mkdir(exist_ok=True)

RESULTS_DIR = Path("results_csv")
RESULTS_DIR.mkdir(exist_ok=True)

COLUMN_NAMES = ["label", "title", "text"]

# Identical to the baseline so the two curves are comparable.
SEED = 42
CHUNK = 100_000
N_CHUNKS = 36
TOTAL_ROWS = 3_600_000
VAL_SIZE = 0.2
# SIZES = [200_000, 500_000] # for checking
SIZES = [700_000, 1_500_000, TOTAL_ROWS] 

# Treat great and GREAT as the same.
LOWERCASE = True
# Tokenize punctuation with its word.
PAD_PUNCT = False
# Combine title+" "+body.
USE_TITLE = True

# fastText defaults for the first run.
FT_PARAMS = dict(
    lr=0.07, # learning rate
    epoch=5, # times we go through training dataset
    wordNgrams=2, # keeping it at 2 as baseline also bi-gram
    dim=10, # vec size
    minCount=1, # to remove junk words/ rare ones that cannot be modeled
    bucket=10_000_000, # I believe this the number of vectors we have in the bank to assinge to n-gram words
    loss="softmax",
    thread=8, # cpu
    seed=42, #reproduciblity
    verbose=2,
)

print(f"fastText {fasttext.__file__}")
print(f"corpora -> {FT_DIR}")

fastText /home/bsarv/.venvs/fasttext/lib/python3.12/site-packages/fasttext/__init__.py
corpora -> /home/bsarv/ft


In [15]:
for lr in (0.5, 0.25, 0.1, 0.05):
    try:
        m = fasttext.train_supervised(input=str(FT_DIR/"train_200000.txt"),
                                      **{**FT_PARAMS, "lr": lr, "epoch": 1})
        print(f"lr={lr}: ok")
    except RuntimeError as e:
        print(f"lr={lr}: {e}")

Read 13M words
Number of words:  458172
Number of labels: 2


lr=0.5: Encountered NaN.


Read 13M words
Number of words:  458172
Number of labels: 2


lr=0.25: Encountered NaN.


Read 13M words
Number of words:  458172
Number of labels: 2
Read 1M words

lr=0.1: Encountered NaN.


Read 13M words

lr=0.05: Encountered NaN.


Read 13M words
Number of words:  458172
Number of labels: 2


In [2]:
# STEP 1 — INGEST
"""preping data: combining body and title"""
def ingest(path, n=None, chunks=False):
    """CSV -> DataFrame[label, combined]. n=None reads the whole file."""
    t0 = time.perf_counter()

    df = pd.read_csv(
        path,
        header=None,
        names=COLUMN_NAMES,
        dtype={"label": "int8"},
        keep_default_na=False,   # empty title stays "", not NaN
        nrows=n,
    )

    if USE_TITLE:
        df["combined"] = (df["title"] + " " + df["text"]).str.strip()
    else:
        df["combined"] = df["text"].str.strip()

    df = df[["label", "combined"]]

    print(
        f"{len(df):,} rows in {time.perf_counter()-t0:.1f}s  "
        f"balance={np.bincount(df['label'].to_numpy())[1:]}  "
        f"memory={df.memory_usage(deep=True).sum()/1e6:.0f} MB"
    )
    return df


peek = ingest(TRAIN_PATH, n=20_000)
peek.head(3)

20,000 rows in 0.2s  balance=[ 9743 10257]  memory=10 MB


,label,combined
0,2,Stuning even for the non-gamer This sound trac...
1,2,The best soundtrack ever to anything. I'm read...
2,2,Amazing! This soundtrack is my favorite music ...


In [3]:
# STEP 2 — NORMALIZE
# cleaning and processing punctuation
_WS    = re.compile(r"\s+")
_PUNCT = re.compile(r"([.,!?;:()\"'])")

def normalize(s):
    """Row-wise cleaning. Must be identical for train, val and test."""
    # s = s.str.replace("__label__", "label", regex=False)  # can't forge a label: scanned code, do not need to clean this
    if PAD_PUNCT:
        s = s.str.replace(_PUNCT, r" \1 ", regex=True)
    s = s.str.replace(_WS, " ", regex=True)               # \n \r \t and runs -> one space
    if LOWERCASE:
        s = s.str.lower()
    return s.str.strip()


def prepare(df):
    """normalize + drop rows that end up empty."""
    df = df.assign(combined=normalize(df["combined"]))
    n0 = len(df)
    df = df[df["combined"].str.len() > 0]
    if n0 != len(df):
        print(f"dropped {n0-len(df):,} empty rows")
    return df

In [4]:
# STEP 3 — SPLIT
#split to test & val
def split(df, seed=SEED):
    return train_test_split(
        df, test_size=VAL_SIZE, random_state=seed, stratify=df["label"]
    )

In [5]:
# STEP 3b — CHECK (on peek)
p = prepare(peek)
tr, va = split(p)
print(len(tr), len(va))
print(p["combined"].str.contains(r"[\n\r\t]").sum(), "rows still holding whitespace chars")
print(repr(p.iloc[1]["combined"][:200]))

16000 4000
0 rows still holding whitespace chars
"the best soundtrack ever to anything. i'm reading a lot of reviews saying that this is the best 'game soundtrack' and i figured that i'd write a review to disagree a bit. this in my opinino is yasunor"


In [6]:
# STEP 4 — WRITE
def write_ft(df, path):
    """DataFrame[label, combined] -> fastText supervised text file."""
    t0 = time.perf_counter()
    lines = "__label__" + df["label"].astype(str) + " " + df["combined"]
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for i in range(0, len(lines), CHUNK):
            f.write("\n".join(lines.iloc[i:i+CHUNK]))
            f.write("\n")
    mb = path.stat().st_size / 1e6 #calc the size in megabytes mb
    print(f"{path.name}: {len(lines):,} lines, {mb:.0f} MB, {time.perf_counter()-t0:.1f}s")

In [7]:
# STEP 5 — BUILD TRAIN/VAL 
def build(size):
    df = prepare(ingest(TRAIN_PATH, n=size))
    tr, va = split(df)
    write_ft(tr, FT_DIR / f"train_{size}.txt")
    write_ft(va, FT_DIR / f"val_{size}.txt")
    del df, tr, va; gc.collect()

for size in SIZES:
    build(size)

700,000 rows in 4.8s  balance=[345693 354307]  memory=341 MB
train_700000.txt: 560,000 lines, 251 MB, 1.7s
val_700000.txt: 140,000 lines, 63 MB, 0.3s
1,500,000 rows in 16.2s  balance=[745397 754603]  memory=728 MB
train_1500000.txt: 1,200,000 lines, 537 MB, 2.5s
val_1500000.txt: 300,000 lines, 134 MB, 0.9s
3,600,000 rows in 58.1s  balance=[1800000 1800000]  memory=1731 MB
train_3600000.txt: 2,880,000 lines, 1275 MB, 27.1s
val_3600000.txt: 720,000 lines, 319 MB, 5.1s


In [8]:
# STEP 6 — BUILD TEST
# test.csv goes through the same prepare() path as train, but is never split — it is the held-out set, used whole.
test_path = FT_DIR / "test.txt"
te = prepare(ingest(TEST_PATH))
write_ft(te, test_path)
del te; gc.collect()

400,000 rows in 4.6s  balance=[200000 200000]  memory=192 MB
test.txt: 400,000 lines, 177 MB, 1.2s


0

In [9]:
# STEP 7 — TRAIN for one size
# SIZE = SIZES[0]
# train_path = FT_DIR / f"train_{SIZE}.txt"
# val_path   = FT_DIR / f"val_{SIZE}.txt"
# assert train_path.exists() and train_path.stat().st_size > 0, train_path
# t0 = time.perf_counter()
# model = fasttext.train_supervised(input=str(train_path), **FT_PARAMS)
# train_secs = time.perf_counter() - t0
# print(f"{SIZE:,}: trained in {train_secs:.1f}s, {len(model.words):,} words, {len(model.labels)} labels")

In [6]:
# STEP 8 — EVAL (val)
def load_ft(path):
    """fastText .txt -> (y, texts). Reads the file the model itself reads, so nothing is normalized twice."""
    y, x = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            lab, _, txt = line.rstrip("\n").partition(" ")
            y.append(int(lab.removeprefix("__label__")))
            x.append(txt)
    return np.array(y, dtype="int8"), x


def evaluate(model, path):
    """-> (y_true, y_pred, predict_seconds)."""
    y, x = load_ft(path)
    t0 = time.perf_counter()
    labs, _ = model.predict(x)
    secs = time.perf_counter() - t0
    yhat = np.array([int(l[0].removeprefix("__label__")) for l in labs], dtype="int8")
    return y, yhat, secs


# y_val, yhat_val, pred_secs = evaluate(model, val_path)
# acc = accuracy_score(y_val, yhat_val)
# f1 = f1_score(y_val, yhat_val, average="macro")
# print(f"{SIZE:,}: val acc={acc:.4f}  macro F1={f1:.4f}  ({len(y_val):,} rows in {pred_secs:.1f}s)")
# print(classification_report(y_val, yhat_val, digits=4))
# print(confusion_matrix(y_val, yhat_val))

# same columns as baseline_full_results.csv so the two curves stack; keyed by size, so a re-run replaces its row
try:
    ft_results
except NameError:
    ft_results = {}
# ft_results[SIZE] = dict(
#     sample_size=SIZE,
#     val_rows=len(y_val),
#     vocab=len(model.words),
#     validation_accuracy=acc,
#     validation_f1=f1,
#     fit_seconds=round(train_secs, 1),
#     predict_seconds=round(pred_secs, 1),
# )
# pd.DataFrame(ft_results.values())

In [ ]:
# # STEP 9 — SWEEP (fit + score, one size at a time)
# try:
#     ft_results
# except NameError:
#     ft_results = {}
    
# for SIZE in SIZES:
#     train_path = FT_DIR / f"train_{SIZE}.txt"
#     val_path   = FT_DIR / f"val_{SIZE}.txt"
#     assert train_path.exists() and train_path.stat().st_size > 0, train_path

#     t0 = time.perf_counter()
#     model = fasttext.train_supervised(input=str(train_path), **FT_PARAMS)
#     train_secs = time.perf_counter() - t0

#     y_val, yhat_val, pred_secs = evaluate(model, val_path)
#     acc = accuracy_score(y_val, yhat_val)
#     f1  = f1_score(y_val, yhat_val, average="macro")
#     print(f"{SIZE:,}: fit {train_secs:.1f}s | val acc={acc:.4f} macroF1={f1:.4f} | {len(model.words):,} words")

#     ft_results[SIZE] = dict(
#         sample_size=SIZE, val_rows=len(y_val), vocab=len(model.words),
#         validation_accuracy=acc, validation_f1=f1,
#         fit_seconds=round(train_secs, 1), predict_seconds=round(pred_secs, 1),
#     )
#     pd.DataFrame(ft_results.values()).to_csv(RESULTS_DIR / "fasttext_results.csv", index=False)

#     del model, y_val, yhat_val; gc.collect()

# pd.DataFrame(ft_results.values())

In [13]:
# STEP 9 — SWEEP (grid: any combination of size + FT_PARAMS overrides)

# Tune on the cheapest size (~12s a fit), then re-run the winner across all SIZES.
TUNE_SIZE = 700_000
BEST_LR = 0.07
GRID = [dict(lr=BEST_LR, epoch=ep) for ep in range(1, 11)]
SWEEP_CSV = RESULTS_DIR / "fasttext_epoch_sweep.csv"

# keyed by (size, config) so re-running a config replaces its row.
# guarded: survives a re-run of this cell, starts clean after a kernel restart.
try:
    ft_sweep
except NameError:
    ft_sweep = {}


def sweep(size, over):
    """Fit + score one config. NaN is caught so one bad config does not kill the grid."""
    train_path = FT_DIR / f"train_{size}.txt"
    val_path   = FT_DIR / f"val_{size}.txt"
    assert train_path.exists() and train_path.stat().st_size > 0, train_path

    params = {**FT_PARAMS, **over}
    label  = ", ".join(f"{k}={v}" for k, v in sorted(over.items())) or "defaults"

    t0 = time.perf_counter()
    try:
        model = fasttext.train_supervised(input=str(train_path), **params)
    except RuntimeError as e:
        print(f"{size:,} | {label}: FAILED — {e}")
        return
    fit_secs = time.perf_counter() - t0

    y_val, yhat_val, pred_secs = evaluate(model, val_path)
    acc = accuracy_score(y_val, yhat_val)
    f1  = f1_score(y_val, yhat_val, average="macro")
    print(f"{size:,} | {label}: acc={acc:.4f} f1={f1:.4f} fit={fit_secs:.1f}s")

    ft_sweep[(size, label)] = dict(
        sample_size=size, config=label,
        lr=params["lr"], dim=params["dim"], epoch=params["epoch"],
        bucket=params["bucket"], wordNgrams=params["wordNgrams"],
        val_rows=len(y_val), vocab=len(model.words),
        validation_accuracy=acc, validation_f1=f1,
        fit_seconds=round(fit_secs, 1), predict_seconds=round(pred_secs, 1),
    )
    pd.DataFrame(ft_sweep.values()).to_csv(SWEEP_CSV, index=False)
    del model, y_val, yhat_val; gc.collect()

BEST = dict(lr=0.07, epoch=5, wordNgrams=3)
for size in SIZES:
    sweep(size, BEST)

pd.DataFrame(ft_sweep.values()).sort_values("validation_accuracy", ascending=False)

Read 45M words
Number of words:  1116466
Number of labels: 2
Progress: 100.0% words/sec/thread: 2872604 lr:  0.000000 avg.loss:  0.146143 ETA:   0h 0m 0s


700,000 | epoch=5, lr=0.07, wordNgrams=3: acc=0.9320 f1=0.9320 fit=15.5s


Read 97M words
Number of words:  1940502
Number of labels: 2
Progress: 100.0% words/sec/thread: 2701509 lr:  0.000000 avg.loss:  0.122275 ETA:   0h 0m 0s 72.8% words/sec/thread: 2733208 lr:  0.019008 avg.loss:  0.151023 ETA:   0h 0m 6s


1,500,000 | epoch=5, lr=0.07, wordNgrams=3: acc=0.9376 f1=0.9376 fit=34.2s


Read 231M words
Number of words:  3629672
Number of labels: 2
Progress: 100.0% words/sec/thread: 2481841 lr:  0.000000 avg.loss:  0.109229 ETA:   0h 0m 0s 0.279461 ETA:   0h 1m10s 71.1% words/sec/thread: 2353591 lr:  0.020264 avg.loss:  0.136944 ETA:   0h 0m17s


3,600,000 | epoch=5, lr=0.07, wordNgrams=3: acc=0.9434 f1=0.9434 fit=93.7s


,sample_size,config,lr,dim,epoch,bucket,wordNgrams,val_rows,vocab,validation_accuracy,validation_f1,fit_seconds,predict_seconds
19,3600000,"epoch=5, lr=0.07, wordNgrams=3",0.07,10,5,10000000,3,720000,3629672,0.943417,0.943417,93.7,70.3
11,3600000,"epoch=5, lr=0.07",0.07,10,5,10000000,2,720000,3629672,0.940712,0.940712,54.5,32.3
18,1500000,"epoch=5, lr=0.07, wordNgrams=3",0.07,10,5,10000000,3,300000,1940502,0.937600,0.937598,34.2,10.8
10,1500000,"epoch=5, lr=0.07",0.07,10,5,10000000,2,300000,1940502,0.935407,0.935405,22.1,5.3
15,700000,wordNgrams=3,0.10,10,5,10000000,3,140000,1116466,0.932836,0.932828,26.0,4.7
17,700000,"epoch=5, lr=0.07, wordNgrams=3",0.07,10,5,10000000,3,140000,1116466,0.932029,0.932019,15.5,4.9
16,700000,wordNgrams=4,0.10,10,5,10000000,4,140000,1116466,0.931886,0.931878,19.7,4.8
14,700000,dim=50,0.10,50,5,10000000,2,140000,1116466,0.929893,0.929882,34.4,13.2
13,700000,dim=20,0.10,20,5,10000000,2,140000,1116466,0.929621,0.929612,14.9,6.6
4,700000,"epoch=5, lr=0.07",0.07,10,5,10000000,2,140000,1116466,0.929371,0.929361,13.2,2.7


In [16]:
FINAL = dict(lr=0.07, epoch=5, wordNgrams=3, minCount=9)
model = fasttext.train_supervised(input=str(FT_DIR / "train_3600000.txt"),
                                  **{**FT_PARAMS, **FINAL})
y_te, yhat_te, secs = evaluate(model, FT_DIR / "test.txt")
print(f"test acc={accuracy_score(y_te, yhat_te):.6f}  "
      f"macroF1={f1_score(y_te, yhat_te, average='macro'):.6f}  ({secs:.1f}s)")
print(classification_report(y_te, yhat_te, digits=4))
model.save_model(str(FT_DIR / "final_3600000.bin"))

Read 231M words
Number of words:  270790
Number of labels: 2
Progress: 100.0% words/sec/thread: 2973362 lr:  0.000000 avg.loss:  0.111379 ETA:   0h 0m 0s 0m47s 14.9% words/sec/thread: 3076473 lr:  0.059564 avg.loss:  0.238739 ETA:   0h 0m40s% words/sec/thread: 3082493 lr:  0.057006 avg.loss:  0.231486 ETA:   0h 0m38s


test acc=0.942203  macroF1=0.942202  (21.4s)
              precision    recall  f1-score   support

           1     0.9426    0.9417    0.9422    200000
           2     0.9418    0.9427    0.9422    200000

    accuracy                         0.9422    400000
   macro avg     0.9422    0.9422    0.9422    400000
weighted avg     0.9422    0.9422    0.9422    400000



In [14]:
# noise floor: identical config, identical seed, 3 runs — only thread scheduling varies
reps = []
for i in range(3):
    m = fasttext.train_supervised(input=str(FT_DIR / "train_700000.txt"),
                                  **{**FT_PARAMS, "lr": 0.07, "epoch": 5, "wordNgrams": 3})
    y, yhat, _ = evaluate(m, FT_DIR / "val_700000.txt")
    reps.append(accuracy_score(y, yhat))
    print(f"rep {i+1}: {reps[-1]:.6f}")
    del m; gc.collect()
print(f"spread = {max(reps) - min(reps):.6f}")

Read 45M words
Number of words:  1116466
Number of labels: 2
Progress: 100.0% words/sec/thread: 2870911 lr:  0.000000 avg.loss:  0.145632 ETA:   0h 0m 0s


rep 1: 0.932264


Read 45M words
Number of words:  1116466
Number of labels: 2
Progress: 100.0% words/sec/thread: 2843405 lr:  0.000000 avg.loss:  0.145148 ETA:   0h 0m 0s lr:  0.029815 avg.loss:  0.201105 ETA:   0h 0m 4s


rep 2: 0.932357


Read 45M words
Number of words:  1116466
Number of labels: 2
Progress: 100.0% words/sec/thread: 2311219 lr:  0.000000 avg.loss:  0.145185 ETA:   0h 0m 0s 0.145185 ETA:   0h 0m 0s


rep 3: 0.932379
spread = 0.000114


In [15]:
# minCount: prunes rare words from the vocab
GRID = [dict(lr=0.07, epoch=5, wordNgrams=3, minCount=mc) for mc in (1, 3, 6, 9)]
for over in GRID:
    sweep(TUNE_SIZE, over)

Read 45M words
Number of words:  1116466
Number of labels: 2
Progress: 100.0% words/sec/thread: 2494090 lr:  0.000000 avg.loss:  0.144717 ETA:   0h 0m 0s


700,000 | epoch=5, lr=0.07, minCount=1, wordNgrams=3: acc=0.9322 f1=0.9321 fit=17.2s


Read 45M words
Number of words:  225252
Number of labels: 2
Progress: 100.0% words/sec/thread: 3269020 lr:  0.000000 avg.loss:  0.144617 ETA:   0h 0m 0s


700,000 | epoch=5, lr=0.07, minCount=3, wordNgrams=3: acc=0.9321 f1=0.9321 fit=13.4s


Read 45M words
Number of words:  128291
Number of labels: 2
Progress: 100.0% words/sec/thread: 2349667 lr:  0.000000 avg.loss:  0.145107 ETA:   0h 0m 0s 3s


700,000 | epoch=5, lr=0.07, minCount=6, wordNgrams=3: acc=0.9319 f1=0.9319 fit=16.3s


Read 45M words
Number of words:  95590
Number of labels: 2
Progress: 100.0% words/sec/thread: 2256675 lr:  0.000000 avg.loss:  0.145390 ETA:   0h 0m 0s


700,000 | epoch=5, lr=0.07, minCount=9, wordNgrams=3: acc=0.9321 f1=0.9320 fit=18.1s
